<a href="https://colab.research.google.com/github/liangchow/zindi-amazon-secret-runway/blob/shruti/model_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Imports & setup

In [1]:
%%capture
!pip -q install rasterio
!pip -q install torch
!pip -q install torchvision

In [2]:
import os
import torch
import numpy as np
from torchvision import transforms
from torch.utils.data import Dataset
from PIL import Image
import rasterio
import matplotlib.pyplot as plt

## Download training data to local compute node

Mount google drive

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Compress training files, copy over and uncompress

In [ ]:
# Navigate to the shared directory
%cd /content/drive/MyDrive/Zindi-Amazon/training
# Zip the data
!zip -r /content/images.zip images
!zip -r /content/masks.zip masks
# Unzip the files
!unzip /content/images.zip -d /content
!unzip /content/masks.zip -d /content

/content/drive/.shortcut-targets-by-id/14mw0v8Bi-MzhsqSI0K3KO23YrUHttM7P/Zindi-Amazon/training
updating: images/ (stored 0%)
updating: images/Sentinel_AllBands_Training_Id_20.tif (deflated 5%)
updating: images/Sentinel_AllBands_Training_Id_59.tif (deflated 5%)
updating: images/Sentinel_AllBands_Training_Id_61.tif (deflated 5%)
updating: images/Sentinel_AllBands_Training_Id_78.tif (deflated 5%)
updating: images/Sentinel_AllBands_Training_Id_79.tif


zip error: Interrupted (aborting)
updating: masks/ (stored 0%)
updating: masks/Mask_Buffer50m_Id_1.tif (deflated 100%)
updating: masks/Mask_Buffer50m_Id_2.tif (deflated 100%)
updating: masks/Mask_Buffer50m_Id_9.tif (deflated 100%)
updating: masks/Mask_Buffer50m_Id_10.tif (deflated 100%)
updating: masks/Mask_Buffer50m_Id_11.tif (deflated 100%)
updating: masks/Mask_Buffer50m_Id_17.tif (deflated 100%)
updating: masks/Mask_Buffer50m_Id_18.tif (deflated 100%)
updating: masks/Mask_Buffer50m_Id_19.tif (deflated 100%)
updating: masks/Mask_Buffer50m_

## Create dataset

In [ ]:
# Custom Dataset class for Sentinel 1/2 bands and mask
class Sentinel2Dataset(Dataset):
    def __init__(self, image_paths, mask_paths, transform=None):
        self.image_paths = image_paths
        self.mask_paths = mask_paths
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        # Load image and mask using rasterio
        image_path = self.image_paths[idx]
        mask_path = self.mask_paths[idx]

        with rasterio.open(image_path) as src:
            # Extract band indexes from descriptions and read directly
            bands = {'B4': None, 'B3': None, 'B2': None}
            for i, desc in enumerate(src.descriptions):
                if desc in bands:
                    bands[desc] = src.read(i + 1)  # Read 1-based bands

            # Check if all required bands were found
            if any(value is None for value in bands.values()):
                raise ValueError(f"Not all bands found in image: {image_path}")

            # Stack the selected bands to form the final image array
            #image = np.stack([bands['B4'], bands['B3'], bands['B2']], axis=-1)
            image = np.stack([np.clip(bands['B4'],0,2000)/2000,
                              np.clip(bands['B3'],0,2000)/2000,
                              np.clip(bands['B2'],0,2000)/2000], axis=-1)

        with rasterio.open(mask_path) as mask_src:
            mask = mask_src.read(1)

        # Ensure both image and mask are numpy arrays before applying transforms
        image = np.array(image)
        mask = np.array(mask)

        # Convert image and mask to PyTorch tensors
        #image = torch.from_numpy(image).type(torch.float32)  # Assuming image is a NumPy array
        #mask = torch.from_numpy(mask).type(torch.float32)   # Assuming mask is a NumPy array

        # Continue with your transformations and other logic:
        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented['image']
            mask = augmented['mask']

        # Only convert to PyTorch tensors if not already tensors (skip if the transform does it)
        if not isinstance(image, torch.Tensor):
            image = torch.from_numpy(image).permute(2, 0, 1).float()  # Channels first
        if not isinstance(mask, torch.Tensor):
            mask = torch.from_numpy(mask).long()

        return image, mask


In [ ]:
image_dir = '/content/images'
mask_dir = '/content/masks'

# List image and mask file paths
image_paths = [os.path.join(image_dir, img) for img in sorted(os.listdir(image_dir)) if img.endswith('.tif')]
mask_paths = [os.path.join(mask_dir, mask) for mask in sorted(os.listdir(mask_dir)) if mask.endswith('.tif')]

## Check Dataset

In [ ]:
# Function to visualize one sample
def visualize_sample(image, mask):
    # Display the image (RGB)
    plt.figure(figsize=(10, 5))

    # Convert image back to (H, W, C) for visualization
    image = np.transpose(image, (1, 2, 0))

    # Plot the RGB image (B2, B3, B4 is in the order: R, G, B)
    plt.subplot(1, 2, 1)
    plt.imshow(image)
    plt.title('Sentinel-2 RGB Image (B2, B3, B4)')

    # Plot the corresponding mask
    plt.subplot(1, 2, 2)
    plt.imshow(mask, cmap='gray')
    plt.title('Airstrip Mask')

    plt.show()

test_dataset = Sentinel2Dataset(image_paths=image_paths, mask_paths=mask_paths)

# Visualize a few samples from the dataset
for i in range(3):
    image, mask = test_dataset[i]
    print(image.shape, mask.shape)
    visualize_sample(image.numpy(), mask.numpy())


## Split into Train, Validation, and Test Sets

In [ ]:
# Create dataset and dataloader

# Apply different transformations to the training, val, and test sets
train_data = Sentinel2Dataset(image_paths=image_paths, mask_paths=mask_paths, transform=get_augmentations(option='train'))
val_data = Sentinel2Dataset(image_paths=image_paths, mask_paths=mask_paths, transform=get_augmentations(option='val'))
test_data =  Sentinel2Dataset(image_paths=image_paths, mask_paths=mask_paths, transform=get_augmentations(option='test'))

# Randomly split the dataset into 80% train / 10% val / 10% test
# by subsetting the transformed train and test datasets
train_size = 0.80
val_size = 0.10
indices = list(range(int(len(train_data))))
train_split = int(train_size * len(train_data))
val_split = int(val_size * len(val_data))
np.random.shuffle(indices)

train_data = data.Subset(train_data, indices=indices[:train_split])
val_data = data.Subset(val_data, indices=indices[train_split: train_split+val_split])
test_data = data.Subset(test_data, indices=indices[train_split+val_split:])
print("Train/val/test sizes: {}/{}/{}".format(len(train_data), len(val_data), len(test_data)))